<a href="https://colab.research.google.com/github/vrockafeller/LabFinal_VR/blob/main/MVP_Stan_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP Stan notebook


`Stan` can run entirely in a notebook like this.

👀 **You can duplicate this notebook to start a new project.**

Note however, that if you use Colab to do work, you will have to spin up a fresh Colab runtime every time you want work on your model and/or the runtime disconnects. Which means downloading CmdStan and compiling your model from scratch. In my own tests, setup takes between 15 and 30 seconds, and compilation of the model takes about 50 seconds. (Your mileage may vary. )

Therefore, **consider a local installation for regular work**, because your Stan isntall will persist and you'll also be able to reuse compiled models between sessions.


**About this notebook**: This notebook installs CmdStanPy, downloads Stan's prebuilt CmdStan package for Google Colab, compiles a minimal model, and generates posterior samples.

If the final cell prints **Stan screen test passed**, your setup passes.

In [ ]:
%pip install -q cmdstanpy==1.3.0

from pathlib import Path
import shutil
import urllib.request

import cmdstanpy
from cmdstanpy import CmdStanModel

CMDSTAN_VERSION = "2.39.0"
cmdstan_dir = Path(f"/content/cmdstan-{CMDSTAN_VERSION}")
archive_path = Path(f"/content/colab-cmdstan-{CMDSTAN_VERSION}.tgz")
archive_url = (
    f"https://github.com/stan-dev/cmdstan/releases/download/"
    f"v{CMDSTAN_VERSION}/colab-cmdstan-{CMDSTAN_VERSION}.tgz"
)

if not cmdstan_dir.exists():
    print(f"Downloading prebuilt CmdStan {CMDSTAN_VERSION} for Colab...")
    urllib.request.urlretrieve(archive_url, archive_path)
    shutil.unpack_archive(archive_path, "/content")

cmdstanpy.set_cmdstan_path(str(cmdstan_dir))
print(f"CmdStanPy {cmdstanpy.__version__}")
print(f"CmdStan {'.'.join(map(str, cmdstanpy.cmdstan_version()))}")

CmdStanPy 1.3.0
CmdStan 2.39


In [ ]:
# Writing a minimal model to a Stan file
stan_code = r"""
data {
  int<lower=0> N;
  int<lower=0, upper=N> K;
}
parameters {
  real<lower=0, upper=1> rho;
}
model {
  rho ~ beta(1.5, 1.5);
  K ~ binomial(N, rho);
}
"""

stan_file = Path("/content/globe_toss.stan")
stan_file.write_text(stan_code)
print(stan_code)


data {
  int<lower=0> N;
  int<lower=0, upper=N> K;
}
parameters {
  real<lower=0, upper=1> rho;
}
model {
  rho ~ beta(1.5, 1.5);
  K ~ binomial(N, rho);
}



In [ ]:
# Testing sampling
model = CmdStanModel(stan_file=str(stan_file))

fit = model.sample(
    data={"N": 7, "K": 3},
    chains=2,
    parallel_chains=2,
    iter_warmup=250,
    iter_sampling=250,
    seed=6300,
    show_progress=False,
)

print("Stan screen test passed.")

Stan screen test passed.
